# HydrAI — Baseline Tree Evaluation (Inlet→Outlet)

Purpose: train and compare default RF, Gradient Boosting, XGBoost, and AdaBoost tree surrogates on exit-plane inlet→outlet data only.

This notebook is intentionally kept simple: no hyperparameter tuning and no full axial-profile training. Use `Main_5_train_evaluate_tune_tree_model_evolution.ipynb` for one-model tuning and full PFR evolution.

Prerequisites: Run Main_2 and Main_3 so `data/processed/features_targets_*.pkl` exists.

---

## Baseline Evaluation Strategy

1. Exit-plane extraction: one sample per simulation run, taken at maximum `relative_position`.
2. Train/Test split: model never sees test data during training.
3. Default tree parameters: compare baseline model families quickly before tuning.
4. Diagnostics: compare Train R² vs Test R² and inspect actual-vs-predicted plots.

---

Sections:
1. Setup & imports
2. Paths & flags
3. Load data
4. Features & targets (exit-plane)
5. Train/test split & scaling
6. Train default models
7. No hyperparameter tuning in Main_4
8. Test-set evaluation & model comparison
9. Actual vs Predicted scatter plots (state variables)
10. Species chemistry analysis (lumped groups)
11. Speed report — Cantera vs ML inference
12. Export baseline models

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. SETUP & IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import os
import sys
import glob
import json
import joblib
import re
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    median_absolute_error,
    max_error,
)
from sklearn.tree import DecisionTreeRegressor
# Project root
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "src").exists() else current_dir.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.utils.plot_style import setup_matplotlib
from src.utils.run_log import start_run_log
from src.ml.dataframe_pickle import load_portable_pickle

setup_matplotlib()
start_run_log('Main_4_train_and_evaluate_tree_models_IO')
print("Libraries imported successfully.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2. PATHS & FLAGS
# ═══════════════════════════════════════════════════════════════════════════════

# I/O flags
IF_PLOT_SHOWN        = True
IF_PLOT_EXPORT       = True
IF_MODEL_EXPORT      = True

# Main_4 is intentionally a simple baseline comparison: default tree settings,
# no hyperparameter tuning, inlet-to-outlet / exit-plane only.
MODELS_TO_TRAIN = ['random_forest', 'gradient_boosting', 'xgboost', 'adaboost']

# Paths
CONFIG_PATH        = project_root / "configs" / "ml" / "ml_training_config.json"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
EXPORT_DIR         = project_root / "models"
FIG_DIR            = project_root / "outputs" / "figures" / "Main_4_train_and_evaluate_tree_models_IO"

print(f"Plot: {IF_PLOT_SHOWN}  |  Export figs: {IF_PLOT_EXPORT}  |  Export models: {IF_MODEL_EXPORT}")
print("Tuning: disabled in Main_4 (use Main_5 for tuned models)")
print(f"Models to evaluate: {MODELS_TO_TRAIN if MODELS_TO_TRAIN else 'all'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3. LOAD CONFIG & DATA
# ═══════════════════════════════════════════════════════════════════════════════

# ── When this notebook actually uses ml_training_config.json ─────────────────
# This cell is the *only* place the JSON file is consulted; nothing else in
# Main_4 re-reads it. It runs on Kernel > Run All, or whenever you re-execute
# this single cell. Keys consumed by Main_4:
#   - top level:           test_size, random_state
#   - random_forest.*      n_estimators, max_depth, min_samples_leaf
#   - gradient_boosting.*  n_estimators, max_depth, min_samples_leaf
#   - xgboost.*            n_estimators, max_depth, learning_rate, reg_alpha, reg_lambda
#   - adaboost.*           n_estimators, learning_rate, max_depth
# If the file is missing, or a key is omitted, the `config.get(..., default)`
# calls below fall back to the inline defaults shown as the second argument.
# The neural_network.* block is read by Main_6 only and is ignored here.
# Edit ml_training_config.json and re-run this cell to change behaviour;
# no kernel restart needed.

if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = json.load(f)
else:
    config = {}
    print(f"[WARN] Config not found: {CONFIG_PATH}. Using defaults.")

TEST_SIZE    = config.get("test_size", 0.2)
RANDOM_STATE = config.get("random_state", 42)
RF_CONFIG    = config.get("random_forest", {"n_estimators": 100, "max_depth": 20})
GB_CONFIG    = config.get("gradient_boosting", {"n_estimators": 150, "max_depth": 5})
XGB_CONFIG   = config.get("xgboost", {"n_estimators": 150, "max_depth": 6})
ADA_CONFIG   = config.get("adaboost", {"n_estimators": 200, "learning_rate": 0.1, "max_depth": 6})

print(f"Config: test_size={TEST_SIZE}, random_state={RANDOM_STATE}")

# Data (latest features_targets_*.pkl)
pkl_files = sorted(glob.glob(str(PROCESSED_DATA_DIR / "features_targets_*.pkl")), reverse=True)
if not pkl_files:
    raise FileNotFoundError(f"No features_targets_*.pkl in {PROCESSED_DATA_DIR}. Run Main_3 first.")

DATA_FILE = pkl_files[0]
loaded = load_portable_pickle(DATA_FILE)
df_features = loaded["df_features"]
df_target = loaded["df_target"]

print(f"Data: {Path(DATA_FILE).name}")
print(f"  Features: {df_features.shape[0]:,} rows × {df_features.shape[1]} cols")
print(f"  Targets:  {df_target.shape[0]:,} rows × {df_target.shape[1]} cols")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4. FEATURES & TARGETS (EXIT-PLANE ONLY)
# ═══════════════════════════════════════════════════════════════════════════════
# For inlet→outlet prediction, we keep only one row per simulation run:
# the row at maximum relative_position (reactor exit).

feature_cols = [
    "initial_temperature_K", "initial_pressure_Pa",
    "reactor_length_m", "reactor_diameter_m",
    "mass_flow_rate_kgps", "heat_flux_Wm2",
    "z_position_m", "relative_position",
]
if "reactant_type" in df_features.columns:
    feature_cols.append("reactant_type")
feature_cols = [c for c in feature_cols if c in df_features.columns]

# Run-identifying columns (exclude spatial coords)
run_cols = [c for c in feature_cols if c not in ("z_position_m", "relative_position")]

# Primary targets (state + thermo)
primary_targets = [
    "temperature_K", "pressure_Pa", "velocity_ms", "density_kgm3",
    "mean_molecular_weight_kgkmol", "heat_capacity_cp_JkgK", "heat_capacity_cv_JkgK",
    "enthalpy_Jkg", "thermal_conductivity_WmK",
]
state_target_cols = [c for c in primary_targets if c in df_target.columns]

# Species targets: mass fractions only — individual Y_* or Y_lump_* from Main_3
_lump_cols = [c for c in df_target.columns if c.startswith('Y_lump_')]
if _lump_cols:
    species_cols = sorted(_lump_cols)
else:
    species_cols = [c for c in df_target.columns if c.startswith('Y_')]

# All targets for training
target_cols = state_target_cols + species_cols

# ═══════════════════════════════════════════════════════════════════════════════
# Chemistry grouping for plots: Y_lump_chem_* / Y_lump_carbon_* map 1:1; else classify Y_*
# ═══════════════════════════════════════════════════════════════════════════════
def classify_species_chemistry(species_name):
    """Classify species by chemical role."""
    name = species_name[2:] if species_name.startswith('Y_') else species_name
    name_base = name.split('(')[0]
    
    if name_base == 'H2':
        return 'hydrogen'
    if name_base in {'Water', 'Ar', 'He', 'Ne', 'N2', 'H'}:
        return 'diluent'
    if name_base == 'C6H14':
        return 'feedstock'
    if name_base in {'C2H4', 'C3H6', 'C4H6', 'C4H8'}:
        return 'olefins'
    if name_base in {'Benzene', 'Toluene', 'Styrene', 'C8H10'}:
        return 'aromatics'
    if name_base in {'CH4', 'CC', 'CCC'}:
        return 'paraffins'
    if name_base.startswith('C#C') or name_base == 'C2H2':
        return 'coke_precursors'
    if name_base in {'C5H6', 'C5H5', 'C3H4', 'C4H4', 'C4H5'}:
        return 'coke_precursors'
    # Highly unsaturated C6+
    match = re.match(r'^C(\d+)H(\d+)', name_base)
    if match:
        c, h = int(match.group(1)), int(match.group(2))
        if c >= 6 and h / c < 1.5:
            return 'coke_precursors'
    # Radicals
    if name_base in {'CH3', 'C2H3', 'C2H5', 'C3H5', 'C3H7', 'C4H7', 'C4H9', 
                     'C5H7', 'C5H9', 'C5H11', 'C6H7', 'C6H9', 'C6H11', 'C6H13',
                     'C7H9', 'C7H11', 'C3H3'}:
        return 'radicals'
    return 'other'

chemistry_groups = {}
if species_cols and all(c.startswith('Y_lump_chem_') for c in species_cols):
    for c in species_cols:
        role = c[len('Y_lump_chem_'):]
        chemistry_groups[role] = [c]
elif species_cols and all(c.startswith('Y_lump_carbon_') for c in species_cols):
    for c in species_cols:
        suf = c[len('Y_lump_carbon_'):]
        # Keep carbon-number labels for Cn buckets, but map inert explicitly to 'inert'.
        role = 'inert' if str(suf).lower() == 'inert' else f'carbon_{suf}'
        chemistry_groups[role] = [c]
else:
    for col in species_cols:
        role = classify_species_chemistry(col)
        chemistry_groups.setdefault(role, []).append(col)

print("Species / lump columns by chemistry role:")
for role, cols in sorted(chemistry_groups.items()):
    print(f"  {role}: {len(cols)} column(s)")

# Extract exit-plane rows (max relative_position per run)
exit_idx = df_features.groupby(run_cols, dropna=False)["relative_position"].idxmax().values
X_exit = df_features.loc[exit_idx, feature_cols].copy()
y_exit = df_target.loc[exit_idx, target_cols].copy()

# Encode categorical (reactant_type)
label_encoder = None
if "reactant_type" in X_exit.columns:
    label_encoder = LabelEncoder()
    X_exit["reactant_type"] = label_encoder.fit_transform(X_exit["reactant_type"].astype(str))

print(f"\nExit-plane dataset: {len(X_exit):,} samples")
print(f"  Input features ({len(feature_cols)}): {feature_cols}")
print(f"  State targets ({len(state_target_cols)}): {state_target_cols}")
print(f"  Species / lump targets: {len(species_cols)} columns")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5. TRAIN/TEST SPLIT & SCALING
# ═══════════════════════════════════════════════════════════════════════════════
# 80/20 split ensures models are evaluated on unseen data.
# StandardScaler normalises features (mean=0, std=1) — fitted on train only.

X_train, X_test, y_train, y_test = train_test_split(
    X_exit, y_exit, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {len(X_train):,} ({100*(1-TEST_SIZE):.0f}%)  |  Test: {len(X_test):,} ({100*TEST_SIZE:.0f}%)")
print(f"Targets: {len(target_cols)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6. TRAIN MODELS (DEFAULT PARAMS)
# ═══════════════════════════════════════════════════════════════════════════════
ALL_MODEL_KEYS = ['random_forest', 'gradient_boosting', 'xgboost', 'adaboost']
keys_to_train = ALL_MODEL_KEYS if MODELS_TO_TRAIN is None else MODELS_TO_TRAIN

models = {}

if 'random_forest' in keys_to_train:
    print("Training Random Forest...")
    base_rf = RandomForestRegressor(
        n_estimators=RF_CONFIG.get("n_estimators", 100),
        max_depth=RF_CONFIG.get("max_depth", 20),
        min_samples_leaf=RF_CONFIG.get("min_samples_leaf", 1),
        random_state=RANDOM_STATE, n_jobs=1
    )
    models['random_forest'] = MultiOutputRegressor(base_rf, n_jobs=-1).fit(X_train_s, y_train)
    print("  Random Forest done.")

if 'gradient_boosting' in keys_to_train:
    print("Training Gradient Boosting...")
    base_gb = GradientBoostingRegressor(
        n_estimators=GB_CONFIG.get("n_estimators", 150),
        max_depth=GB_CONFIG.get("max_depth", 5),
        min_samples_leaf=GB_CONFIG.get("min_samples_leaf", 1),
        random_state=RANDOM_STATE
    )
    models['gradient_boosting'] = MultiOutputRegressor(base_gb, n_jobs=-1).fit(X_train_s, y_train)
    print("  Gradient Boosting done.")

if 'xgboost' in keys_to_train:
    print("Training XGBoost...")
    base_xgb = xgb.XGBRegressor(
        n_estimators=XGB_CONFIG.get("n_estimators", 150),
        max_depth=XGB_CONFIG.get("max_depth", 6),
        learning_rate=XGB_CONFIG.get("learning_rate", 0.1),
        reg_alpha=XGB_CONFIG.get("reg_alpha", 0),
        reg_lambda=XGB_CONFIG.get("reg_lambda", 1),
        random_state=RANDOM_STATE, n_jobs=1
    )
    models['xgboost'] = MultiOutputRegressor(base_xgb, n_jobs=-1).fit(X_train_s, y_train)
    print("  XGBoost done.")

if 'adaboost' in keys_to_train:
    print("Training AdaBoost...")
    base_ada = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(
            max_depth=ADA_CONFIG.get("max_depth", 6),
            random_state=RANDOM_STATE
        ),
        n_estimators=ADA_CONFIG.get("n_estimators", 200),
        learning_rate=ADA_CONFIG.get("learning_rate", 0.1),
        random_state=RANDOM_STATE
    )
    models['adaboost'] = MultiOutputRegressor(base_ada, n_jobs=-1).fit(X_train_s, y_train)
    print("  AdaBoost done.")

print(f"\nTrained models: {list(models.keys())}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7. NO HYPERPARAMETER TUNING IN MAIN_4
# ═══════════════════════════════════════════════════════════════════════════════
# Main_4 is the fast baseline notebook: default tree models, exit-plane only.
# Use Main_5_train_evaluate_tune_tree_model_evolution.ipynb for tuned one-model
# workflows, including full axial/PFR evolution.

print("Main_4 uses default model parameters only. No hyperparameter tuning is run here.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8. TEST-SET EVALUATION & MODEL COMPARISON
# ═══════════════════════════════════════════════════════════════════════════════
# Key metrics:
#   R² — explained variance (1.0 = perfect, <0 = worse than mean)
#   MAE — mean absolute error (interpretable units)
#   RMSE — root mean squared error (penalises large errors)
#   MAPE% — mean absolute percentage error (scale-independent)
#   MBE — mean bias error (positive = overprediction)
#
# Overfitting check: Train R² >> Test R² indicates overfitting.

DISPLAY_NAMES = {
    'random_forest': 'Random Forest',
    'gradient_boosting': 'Gradient Boosting',
    'xgboost': 'XGBoost',
    'adaboost': 'AdaBoost',
}

def compute_metrics(y_true, y_pred, target_names):
    """Compute per-target metrics."""
    rows = []
    for i, tgt in enumerate(target_names):
        yt = y_true[:, i] if y_true.ndim > 1 else y_true
        yp = y_pred[:, i] if y_pred.ndim > 1 else y_pred
        mask = np.abs(yt) > 1e-12
        mape = np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100 if mask.any() else np.nan
        rows.append({
            'target': tgt,
            'R2': r2_score(yt, yp),
            'MAE': mean_absolute_error(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAPE_%': mape,
            'MBE': np.mean(yp - yt),
        })
    return pd.DataFrame(rows)

# Compute metrics for each model
results = []
for key, model in models.items():
    # Train metrics (for overfitting check)
    y_train_pred = model.predict(X_train_s)
    train_r2 = r2_score(y_train, y_train_pred, multioutput='uniform_average')
    
    # Test metrics
    y_test_pred = model.predict(X_test_s)
    test_r2 = r2_score(y_test, y_test_pred, multioutput='uniform_average')
    test_mae = mean_absolute_error(y_test, y_test_pred, multioutput='uniform_average')
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred, multioutput='uniform_average'))
    
    # Per-target MAPE
    y_test_np = y_test.values if hasattr(y_test, 'values') else y_test
    mask = np.abs(y_test_np) > 1e-12
    mape_per_target = np.abs((y_test_np - y_test_pred) / np.where(mask, y_test_np, 1)) * 100
    mape_per_target = np.where(mask, mape_per_target, np.nan)
    test_mape = np.nanmean(mape_per_target)
    
    results.append({
        'Model': DISPLAY_NAMES.get(key, key),
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_RMSE': test_rmse,
        'Test_MAPE_%': test_mape,
        'Overfit_Gap': train_r2 - test_r2,
    })

df_results = pd.DataFrame(results).sort_values('Test_R2', ascending=False)

def _extract_best_model_info(best_model_obj):
    """Return compact info for the best model (counts + key hyperparameters)."""
    info = {
        'base_estimator': type(best_model_obj).__name__,
        'n_outputs': None,
        'n_total_estimators': None,
        'key_params': {},
    }

    # Most models here are wrapped in MultiOutputRegressor
    if hasattr(best_model_obj, 'estimators_') and len(best_model_obj.estimators_) > 0:
        first_est = best_model_obj.estimators_[0]
        info['base_estimator'] = type(first_est).__name__
        info['n_outputs'] = len(best_model_obj.estimators_)

        if hasattr(first_est, 'n_estimators'):
            n_per_output = getattr(first_est, 'n_estimators', None)
            if isinstance(n_per_output, (int, np.integer)):
                info['n_total_estimators'] = int(n_per_output) * info['n_outputs']

        params = first_est.get_params() if hasattr(first_est, 'get_params') else {}
        key_fields = [
            'n_estimators', 'max_depth', 'learning_rate', 'min_samples_split',
            'min_samples_leaf', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda'
        ]
        info['key_params'] = {k: params[k] for k in key_fields if k in params}

    return info

print("=" * 80)
print("MODEL COMPARISON — Test Set Evaluation")
print("=" * 80)
print(df_results.to_string(index=False))
print()
print("Key: Overfit_Gap = Train_R² - Test_R² (large gap > 0.05 suggests overfitting)")
print("=" * 80)

# Best model summary with additional context
best_display = str(df_results.iloc[0]['Model'])
best_key = next((k for k, v in DISPLAY_NAMES.items() if v == best_display), None)
if best_key is not None and best_key in models:
    best_info = _extract_best_model_info(models[best_key])

    n_train = X_train.shape[0] if hasattr(X_train, 'shape') else len(X_train)
    n_test = X_test.shape[0] if hasattr(X_test, 'shape') else len(X_test)
    n_features = X_train.shape[1] if hasattr(X_train, 'shape') and len(X_train.shape) > 1 else None
    n_targets = y_train.shape[1] if hasattr(y_train, 'shape') and len(y_train.shape) > 1 else 1

    print(f"Best model: {best_display}")
    print(f"  Data: n_train={n_train:,}, n_test={n_test:,}, n_features={n_features}, n_targets={n_targets}")
    print(f"  Base estimator: {best_info['base_estimator']}")
    if best_info['n_outputs'] is not None:
        print(f"  Multi-output estimators: {best_info['n_outputs']}")
    if best_info['n_total_estimators'] is not None:
        print(f"  Approx total trees: {best_info['n_total_estimators']:,}")
    if best_info['key_params']:
        print(f"  Key hyperparameters: {best_info['key_params']}")
    print("=" * 80)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8b. PER-TARGET METRICS TABLE
# ═══════════════════════════════════════════════════════════════════════════════
# Detailed breakdown per target variable.

per_target_rows = []
for key, model in models.items():
    y_pred = model.predict(X_test_s)
    y_test_np = y_test.values if hasattr(y_test, 'values') else y_test
    for i, tgt in enumerate(target_cols):
        yt = y_test_np[:, i]
        yp = y_pred[:, i]
        mask = np.abs(yt) > 1e-12
        mape = np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100 if mask.any() else np.nan
        per_target_rows.append({
            'Model': DISPLAY_NAMES.get(key, key),
            'Target': tgt,
            'R2': r2_score(yt, yp),
            'MAE': mean_absolute_error(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAPE_%': mape,
        })

df_per_target = pd.DataFrame(per_target_rows)

# Pivot for comparison
print("\nPer-target R² (Test Set):")
print("-" * 60)
pivot_r2 = df_per_target.pivot(index='Target', columns='Model', values='R2')
print(pivot_r2.round(4).to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9. ACTUAL VS PREDICTED — SCATTER PLOTS (STATE VARIABLES ONLY)
# ═══════════════════════════════════════════════════════════════════════════════
# Perfect predictions lie on y=x diagonal. Spread around diagonal = error.
# Only plotting state targets here; species are analyzed in chemistry section.

# Only state variables for scatter plots (species handled in chemistry section)
scatter_targets = state_target_cols
n_scatter = len(scatter_targets)
n_models = len(models)

if n_scatter > 0 and n_models > 0:
    fig, axes = plt.subplots(n_scatter, n_models, figsize=(4 * n_models, 3.5 * n_scatter), squeeze=False)
    
    for j, (key, model) in enumerate(models.items()):
        y_pred = model.predict(X_test_s)
        y_test_np = y_test.values if hasattr(y_test, 'values') else y_test
        
        for i, tgt in enumerate(scatter_targets):
            ax = axes[i, j]
            tgt_idx = target_cols.index(tgt)
            yt = y_test_np[:, tgt_idx]
            yp = y_pred[:, tgt_idx]

            tgt_label = tgt
            if tgt == 'pressure_Pa':
                yt = yt / 1e5
                yp = yp / 1e5
                tgt_label = 'Pressure (bar)'
            
            ax.scatter(yt, yp, alpha=0.25, s=10, edgecolors='none', c='b')
            lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
            ax.plot(lims, lims, 'r-', lw=2, label='y=x')
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            
            r2 = r2_score(yt, yp)
            ax.set_title(f"{DISPLAY_NAMES.get(key, key)}\n{tgt_label}", fontsize=9)
            ax.text(0.05, 0.95, f"R²={r2:.4f}", transform=ax.transAxes, fontsize=8, va='top')
            
            if i == n_scatter - 1:
                ax.set_xlabel('Actual')
            if j == 0:
                ax.set_ylabel('Predicted')
    
    plt.suptitle('Actual vs Predicted — State Variables (Test Set)', y=1.01)
    plt.tight_layout()
    
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / 'actual_vs_predicted_state_scatter.png', dpi=150, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)
else:
    print("No state targets or models to plot.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 10. SPECIES CHEMISTRY ANALYSIS (LUMPED GROUPS)
# ═══════════════════════════════════════════════════════════════════════════════
# Analyze model performance on chemistry-grouped species.
# Groups: olefins (primary products), aromatics (BTX), paraffins, coke_precursors, radicals, etc.

# Use the best model for species analysis
best_model_key = df_results.iloc[0]['Model'].lower().replace(' ', '_')
if best_model_key not in models:
    best_model_key = list(models.keys())[0]
best_model = models[best_model_key]
print(f"Using best model for species analysis: {DISPLAY_NAMES.get(best_model_key, best_model_key)}")

# Get predictions
y_test_pred = best_model.predict(X_test_s)
y_test_np = y_test.values if hasattr(y_test, 'values') else y_test

# ── Plot 1: Lumped yields comparison (actual vs predicted) ──────────────────────
_default_chem_order = ['olefins', 'aromatics', 'paraffins', 'coke_precursors', 'radicals', 'feedstock', 'hydrogen', 'diluent', 'other']
if chemistry_groups and all(k in _default_chem_order for k in chemistry_groups.keys()):
    chemistry_order = [c for c in _default_chem_order if c in chemistry_groups]
else:
    chemistry_order = sorted(chemistry_groups.keys())

# Calculate lumped yields for each chemistry group
lumped_actual = {}
lumped_pred = {}
for role in chemistry_order:
    cols = chemistry_groups[role]
    col_idx = [target_cols.index(c) for c in cols if c in target_cols]
    if col_idx:
        lumped_actual[role] = y_test_np[:, col_idx].sum(axis=1).mean()
        lumped_pred[role] = y_test_pred[:, col_idx].sum(axis=1).mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(lumped_actual))
width = 0.35
ax.bar(x - width/2, list(lumped_actual.values()), width, label='Actual', color='b', alpha=0.8)
ax.bar(x + width/2, list(lumped_pred.values()), width, label='Predicted (best model)', color='red', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(list(lumped_actual.keys()), rotation=0, ha='center')
ax.set_ylabel('Mean lumped mass fraction (Y)')
ax.set_title(f'Chemistry-grouped yields: Actual vs Predicted (Test Set)\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}')
ax.legend()
plt.tight_layout()
if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'chemistry_lumped_yields_comparison.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

# ── Plot 2: Normalized MAE per chemistry group ─────────────────────────────────
#   [Normalized Mean Absolute Error (NMAE) in chemistry applications is often used to 
#   compare model performance across different chemical groups with varying scales]
nmae_by_group = {}
for role in chemistry_order:
    cols = chemistry_groups[role]
    col_idx = [target_cols.index(c) for c in cols if c in target_cols]
    if col_idx:
        yt = y_test_np[:, col_idx].sum(axis=1)
        yp = y_test_pred[:, col_idx].sum(axis=1)

        mae = mean_absolute_error(yt, yp)
        yrange = np.max(yt) - np.min(yt)
        if yrange > 1e-12:
            nmae = mae / yrange
        else:
            # Fallback for near-constant targets
            ymean = np.mean(np.abs(yt))
            nmae = mae / ymean if ymean > 1e-12 else 0.0
        nmae_by_group[role] = nmae * 100.0  # percentage

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(nmae_by_group.keys(), nmae_by_group.values(), color='gray', edgecolor='white', linewidth=0.5)
ax.axhline(5, color='g', linestyle='--', alpha=1, label='NMAE <= 5%: excellent')
ax.axhline(10, color='b', linestyle='--', alpha=1, label='NMAE <= 10%: good')
ax.axhline(20, color='r', linestyle='--', alpha=1, label='NMAE > 20%: weak')
ax.set_ylabel('Normalized MAE (%)')
ax.set_xlabel('Chemistry group')
ax.set_title(f'Model error by chemistry group (Test Set)\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'chemistry_group_nmae.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

# ── Plot 3: Normalized MAE for state / thermo / aero targets ───────────────────
state_nmae = {}
for tgt in state_target_cols:
    tgt_idx = target_cols.index(tgt)
    yt = y_test_np[:, tgt_idx]
    yp = y_test_pred[:, tgt_idx]

    mae = mean_absolute_error(yt, yp)
    yrange = np.max(yt) - np.min(yt)
    if yrange > 1e-12:
        nmae = mae / yrange
    else:
        ymean = np.mean(np.abs(yt))
        nmae = mae / ymean if ymean > 1e-12 else 0.0
    state_nmae[tgt] = nmae * 100.0

_state_labels = {
    'temperature_K': 'Exit T',
    'pressure_Pa': 'Exit P (bar)',
    'velocity_ms': 'Velocity',
    'density_kgm3': 'Density',
    'mean_molecular_weight_kgkmol': 'MW',
    'heat_capacity_cp_JkgK': 'Cp',
    'heat_capacity_cv_JkgK': 'Cv',
    'enthalpy_Jkg': 'Enthalpy',
    'thermal_conductivity_WmK': 'Therm. cond.',
}

fig, ax = plt.subplots(figsize=(10, 4.5))
labels = [_state_labels.get(t, t) for t in state_nmae.keys()]
ax.bar(labels, state_nmae.values(), color='slateblue', edgecolor='white', linewidth=0.5, alpha=0.85)
ax.axhline(5, color='g', linestyle='--', alpha=1, label='NMAE <= 5%: excellent')
ax.axhline(10, color='b', linestyle='--', alpha=1, label='NMAE <= 10%: good')
ax.axhline(20, color='r', linestyle='--', alpha=1, label='NMAE > 20%: weak')
ax.set_ylabel('Normalized MAE (%)')
ax.set_xlabel('Exit state / thermo target')
ax.set_title(f'Error by exit state / thermo target (Test Set)\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'state_thermo_target_nmae.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

print("\nSTATE / THERMO TARGET ERROR (Normalized MAE %, Test Set)")
df_state_nmae = pd.DataFrame({
    'Target': list(state_nmae.keys()),
    'NMAE_%': list(state_nmae.values()),
})
print(df_state_nmae.to_string(index=False))

# ── Plot 4: Scatter plot — actual vs predicted for chemistry groups ────────────
# Single non-duplicative view: all available chemistry/carbon groups (dynamic grid).
all_groups = [g for g in chemistry_order if g in chemistry_groups]
if all_groups:
    n_panels = len(all_groups)
    n_cols = min(4, n_panels)
    n_rows = (n_panels + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 3.6 * n_rows), squeeze=False)
    flat_axes = axes.ravel()

    for i, role in enumerate(all_groups):
        ax = flat_axes[i]
        cols = chemistry_groups[role]
        col_idx = [target_cols.index(c) for c in cols if c in target_cols]
        if col_idx:
            yt = y_test_np[:, col_idx].sum(axis=1)
            yp = y_test_pred[:, col_idx].sum(axis=1)
            ax.scatter(yt, yp, alpha=0.25, s=10, edgecolors='none', c='b')
            lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
            ax.plot(lims, lims, 'r-', lw=1.5)
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            r2 = r2_score(yt, yp)
            ax.set_title(f'{role.replace("_", " ").title()}\nR²={r2:.3f}', fontsize=9)
            ax.set_xlabel('Actual', fontsize=8)
            ax.set_ylabel('Predicted', fontsize=8)

    for j in range(n_panels, len(flat_axes)):
        flat_axes[j].set_visible(False)

    plt.suptitle(f'Lumped yields: Actual vs Predicted (All groups)\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}', y=1.02)
    plt.tight_layout()
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / 'chemistry_scatter_all_groups.png', dpi=150, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)

# ── Plot 5: Best-model state targets (exit predictions) ───────────────────────
preferred_state_targets = ['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3']
state_scatter_targets = [t for t in preferred_state_targets if t in state_target_cols]
if not state_scatter_targets:
    state_scatter_targets = state_target_cols[:4]

if state_scatter_targets:
    n_panels = len(state_scatter_targets)
    n_cols = min(4, n_panels)
    n_rows = (n_panels + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 4.2 * n_rows), squeeze=False)
    flat_axes = axes.ravel()

    for i, tgt in enumerate(state_scatter_targets):
        ax = flat_axes[i]
        tgt_idx = target_cols.index(tgt)
        yt = y_test_np[:, tgt_idx]
        yp = y_test_pred[:, tgt_idx]

        if tgt == 'pressure_Pa':
            yt = yt / 1e5
            yp = yp / 1e5
            unit_label = 'Pressure (bar)'
        elif tgt == 'temperature_K':
            unit_label = 'Temperature (K)'
        elif tgt == 'velocity_ms':
            unit_label = 'Velocity (m/s)'
        elif tgt == 'density_kgm3':
            unit_label = 'Density (kg/m³)'
        else:
            unit_label = tgt

        ax.scatter(yt, yp, alpha=0.25, s=10, edgecolors='none', c='b')
        lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        ax.plot(lims, lims, 'r-', lw=1.5)
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        r2 = r2_score(yt, yp)
        ax.set_title(f'{unit_label}\nR²={r2:.4f}', fontsize=10)
        ax.set_xlabel('Actual')
        ax.set_ylabel('Predicted')

    for j in range(n_panels, len(flat_axes)):
        flat_axes[j].set_visible(False)

    plt.suptitle(f'Best-model exit state predictions\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}', y=1.02)
    plt.tight_layout()
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / 'state_scatter_best_model_exit.png', dpi=150, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)

# ── Summary table ───────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("CHEMISTRY GROUP PERFORMANCE (lumped mass fractions)")
print("=" * 60)
summary_rows = []
for role in chemistry_order:
    if role in lumped_actual and role in nmae_by_group:
        summary_rows.append({
            'Group': role,
            'N_species': len(chemistry_groups.get(role, [])),
            'Actual_Y': lumped_actual[role],
            'Predicted_Y': lumped_pred[role],
            'Error_%': (lumped_pred[role] - lumped_actual[role]) / lumped_actual[role] * 100 if lumped_actual[role] > 1e-12 else 0,
            'NMAE_%': nmae_by_group[role],
        })
df_chem = pd.DataFrame(summary_rows)
print(df_chem.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11. SPEED REPORT — CANTERA VS ML INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════
# Set this from a measured Cantera/PFR timing if available, e.g. from Main_1 or Main_2 logs.
# It should represent seconds for one full Cantera PFR simulation to the outlet.
if 'CANTERA_EXIT_SECONDS_PER_RUN' not in globals():
    CANTERA_EXIT_SECONDS_PER_RUN = None  # example: 2.5

best_speed_key = df_results.iloc[0]['Model'].lower().replace(' ', '_')
if best_speed_key not in models:
    best_speed_key = list(models.keys())[0]
best_speed_model = models[best_speed_key]

# Warm-up avoids one-time overhead in the timed loop.
_ = best_speed_model.predict(X_test_s[: min(10, len(X_test_s))])

n_repeats = 5
start = time.perf_counter()
for _ in range(n_repeats):
    _ = best_speed_model.predict(X_test_s)
elapsed = time.perf_counter() - start

ml_batch_seconds = elapsed / n_repeats
ml_seconds_per_exit = ml_batch_seconds / len(X_test_s)
ml_exits_per_second = len(X_test_s) / ml_batch_seconds

speed_report = {
    'model': DISPLAY_NAMES.get(best_speed_key, best_speed_key),
    'n_test_samples': len(X_test_s),
    'ml_batch_seconds': ml_batch_seconds,
    'ml_seconds_per_exit_prediction': ml_seconds_per_exit,
    'ml_exit_predictions_per_second': ml_exits_per_second,
    'cantera_seconds_per_exit_run': CANTERA_EXIT_SECONDS_PER_RUN,
    'speedup_vs_cantera': None,
}

print(
    f"Speed (exit) | {speed_report['model']} | n={len(X_test_s)} | "
    f"batch {ml_batch_seconds:.3f}s | {ml_seconds_per_exit * 1e3:.2f} ms/exit | "
    f"{ml_exits_per_second:.0f} exits/s"
)
if CANTERA_EXIT_SECONDS_PER_RUN is not None and CANTERA_EXIT_SECONDS_PER_RUN > 0:
    speedup = CANTERA_EXIT_SECONDS_PER_RUN / ml_seconds_per_exit
    speed_report['speedup_vs_cantera'] = speedup
    print(f"  vs Cantera {CANTERA_EXIT_SECONDS_PER_RUN:.3f}s/run → ~{speedup:.0f}x")
else:
    print("  (Set CANTERA_EXIT_SECONDS_PER_RUN for speedup vs Cantera)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 12. EXPORT MODELS
# ═══════════════════════════════════════════════════════════════════════════════
# Save trained models, scaler, and metadata for inference.

if IF_MODEL_EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    run_at = datetime.now().isoformat(timespec="seconds")
    artifact_path = EXPORT_DIR / "tree_models_exit.joblib"
    
    artifact = {
        'models': models,
        'scaler_X': scaler,
        'feature_cols': feature_cols,
        'target_cols': target_cols,
        'label_encoder': label_encoder,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'X_train_s': X_train_s,
        'X_test_s': X_test_s,
        'config': config,
        'data_file': DATA_FILE,
        'run_at': run_at,
    }
    if 'speed_report' in globals():
        artifact['speed_report'] = speed_report
    
    joblib.dump(artifact, artifact_path)
    print(f"Exported {artifact_path.name} | models={list(models.keys())} | "
          f"feat={len(feature_cols)} tgt={len(target_cols)}")
else:
    print("Model export disabled.")

---

## Summary

This notebook evaluates default tree baselines for inlet→outlet (exit-plane) prediction:

1. Data: Exit-plane samples extracted from full PFR profiles.
2. Models: RF, Gradient Boosting, XGBoost, and optionally AdaBoost.
3. No tuning: default parameters only, for fast model-family comparison.
4. Evaluation: R², MAE, RMSE, MAPE on a held-out test set.
5. Chemistry analysis: lumped species/yield diagnostics with Normalized MAE by group.
6. Speed report: ML inference timing, with optional Cantera/PFR speedup when `CANTERA_EXIT_SECONDS_PER_RUN` is set.

Use `Main_5_train_evaluate_tune_tree_model_evolution.ipynb` for one-model hyperparameter tuning and full axial/PFR evolution.